# 04. Análise estatística segundo Demšar (2006) e análise por regime

Roteiro da Etapa 2. **Este notebook só é executado depois de `python -m src.pipeline.final_test`**: ele lê `results/test.csv`, gerado uma única vez após o congelamento do protocolo (`results/protocolo.lock`). Nada do que aparece aqui pode retroalimentar o modelo, o espaço de busca ou a seleção de datasets.

O notebook usa as funções que o grupo implementa em `src/pipeline/demsar.py`, `src/pipeline/bootstrap.py`, `src/pipeline/regime.py` e `src/reports/`. Enquanto uma função não estiver implementada, a célula correspondente falha com `NotImplementedError`; os testes em `tests/aceitacao/` indicam quando cada função está correta.

Protocolo (Demšar, 2006):
1. uma medida por dataset: a média dos 3 folds externos oficiais (folds não são observações independentes);
2. postos por dataset e teste de Friedman com a correção de Iman-Davenport (decisão por F_F);
3. modelo do grupo (com busca) como controle: Holm sobre os z dos postos médios e diagrama de Bonferroni-Dunn;
4. análise complementar todos contra todos: Nemenyi e diagrama CD;
5. padrão vs. com busca: Wilcoxon signed-ranks.

As respostas às perguntas devem constar no relatório.

In [ ]:
import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

from src.pipeline import demsar as D
from src.pipeline.bootstrap import bootstrap_table
from src.pipeline.evaluate import HIGHER_IS_BETTER
from src.pipeline.regime import REGIME_COLUMNS, aggregate_by_regime, rank_by_regime, regime_metadata
from src.pipeline.results import load_raw, per_dataset
from src.reports.plots import boxplot_by_model, cost_vs_performance
from src.reports.results_table import per_dataset_table, summary_by_model

RESULTS = REPO / 'results'
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
METRIC = 'auc_ovo'          # métrica primária (declarada antes da análise)
ALPHA = 0.05
CONTROL = 'group_tuned'     # controle: modelo do grupo após a busca
SYSTEMS = [CONTROL, 'lightgbm', 'xgboost', 'catboost']
HIB = HIGHER_IS_BETTER[METRIC]

In [ ]:
raw = load_raw(RESULTS / 'test.csv')   # avaliação final única (final_test)
print(raw['status'].value_counts())
raw[raw['status'] != 'ok'][['dataset', 'fold', 'model', 'error']]

## 1. Tabelas descritivas

In [ ]:
summary_by_model(raw)

In [ ]:
per_dataset_table(raw, METRIC)

## 2. Teste de Friedman com correção de Iman-Davenport

In [ ]:
table = per_dataset(raw, METRIC, models=SYSTEMS)
N, k = table.shape
print(f'N = {N} datasets completos, k = {k} sistemas')
fr = D.friedman_test(table, higher_is_better=HIB)
print(fr.avg_ranks.sort_values())
print(f'chi2_F = {fr.chi2_f:.3f} (p = {fr.p_value_chi2:.4g})')
print(f'F_F = {fr.f_f:.3f}, gl = ({fr.df1}, {fr.df2}), p = {fr.p_value_ff:.4g}')

**Pergunta 1.** A hipótese nula do teste de Friedman é rejeitada a 5%? Por que a decisão é tomada por F_F e não por chi2_F? Quantos datasets ficaram de fora pela regra de completude e por quê?

## 3. Modelo do grupo como controle: Holm e Bonferroni-Dunn

In [ ]:
holm_table = D.holm_vs_control(fr.avg_ranks, N, control=CONTROL, alpha=ALPHA)
holm_table

In [ ]:
cd_bd = D.critical_difference(k, N, ALPHA, method='bonferroni-dunn')
fig = D.plot_cd_diagram(fr.avg_ranks, cd_bd, control=CONTROL, title=f'Bonferroni-Dunn ({METRIC})')
fig.savefig(FIGURES / f'cd_bonferroni_dunn_{METRIC}.png', dpi=150, bbox_inches='tight')

**Pergunta 2.** Contra quais baselines o modelo do grupo difere significativamente? As conclusões por Holm e por Bonferroni-Dunn coincidem? Qual dos dois procedimentos é mais poderoso, segundo Demšar (2006)?

## 4. Análise complementar: Nemenyi e diagrama CD

In [ ]:
cd_nem = D.critical_difference(k, N, ALPHA, method='nemenyi')
print(f'CD (Nemenyi) = {cd_nem:.3f}')
display(D.nemenyi_pairs(fr.avg_ranks, N, ALPHA))
fig = D.plot_cd_diagram(fr.avg_ranks, cd_nem, title=f'Nemenyi ({METRIC})')
fig.savefig(FIGURES / f'cd_nemenyi_{METRIC}.png', dpi=150, bbox_inches='tight')

**Pergunta 3.** Por que o artigo recomenda não usar comparações todos contra todos quando a pergunta é se um método supera os existentes? Como interpretar um sistema que pertence a dois grupos sobrepostos no diagrama?

## 5. Padrão vs. com busca (Wilcoxon) e robustez par a par

**Nota.** O Wilcoxon entre `group_tuned` e `group_default` sobre o teste é **descritivo** do resultado final: ele não serve para decidir se a busca deve ser usada nem para alterar o espaço de busca. Os sistemas e o controle `group_tuned` foram fixados a priori pelo enunciado; `results/validation.csv` é apenas diagnóstico (o treino de um fold externo contém o teste dos outros) e não foi base dessa escolha.

In [ ]:
pair = per_dataset(raw, METRIC, models=['group_tuned', 'group_default'])
D.wilcoxon_signed_ranks(pair['group_tuned'], pair['group_default'], higher_is_better=HIB)

In [ ]:
D.wilcoxon_vs_control(table, CONTROL, higher_is_better=HIB, alpha=ALPHA)

**Pergunta 4.** A busca de hiperparâmetros melhorou o modelo de forma significativa no teste? O ganho compensa o tempo de busca (ver seção 7)? O resultado confirma o que a validação interna indicava? As conclusões do Wilcoxon com Holm coincidem com as da seção 3? Se não, discuta (Benavoli, Corani e Mangili, 2016).

## 6. Incerteza no conjunto de teste (bootstrap)

Cada intervalo refere-se a um dataset e fold. A média dos limites entre datasets **não** é um intervalo de confiança da média entre datasets; o resumo usa a **mediana da largura** dos intervalos por sistema. A tabela completa vai para o apêndice do relatório.

**Tempo.** Com `n_boot=1000`, o cálculo percorre todos os arquivos de `results/preds/` (datasets, folds e sistemas) e pode levar de minutos a algumas horas. Grave o resultado para não repetir o cálculo, por exemplo `ci.to_csv(FIGURES / 'bootstrap_ci.csv', index=False)`. Não reduza `n_boot`.

In [ ]:
ci = bootstrap_table(RESULTS / 'preds', metric=METRIC, n_boot=1000)
ci['width'] = ci['ci_high'] - ci['ci_low']
ci.groupby('model')['width'].agg(['median', 'min', 'max'])   # resumo: mediana da largura

## 7. Regimes e custo computacional (descritivo)

In [ ]:
meta = regime_metadata(table.index.tolist())
for col in REGIME_COLUMNS:
    print(f'\n=== {col} ===')
    print(aggregate_by_regime(table, meta, col).to_string(index=False))
rank_by_regime(table, meta, 'regime_size', higher_is_better=HIB)

In [ ]:
boxplot_by_model(table, METRIC, output=FIGURES / f'boxplot_{METRIC}.png');

In [ ]:
cost_vs_performance(raw, METRIC, output=FIGURES / 'custo_vs_desempenho.png');

**Pergunta 5.** Em quais regimes o modelo do grupo se destaca ou falha? Por que essa análise é apenas descritiva (considere o número de datasets por célula e a CD com N = 10)? Qual sistema oferece a melhor relação entre custo e desempenho?

**Pergunta 6.** Repita as seções 2 a 5 para uma métrica secundária (por exemplo, entropia cruzada, em que menor é melhor). As conclusões mudam? Como tratar a multiplicidade entre métricas?